# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring

**Decision:** At the end of March 2026, help editors prioritize content items for review. This supports decision-making; it does not prove that refreshing content will increase traffic.

**Data access:** This notebook uses the gated FlyRank warehouse. Request access, create a Hugging Face Read token, and store it in Google Colab Secrets as `HF_TOKEN`. Never expose the token in notebook cells or commit it to GitHub.

**Execution:** Run all cells in order. The queries use the March and April 2026 data partitions.


## 1. The contract in plain words

1. **One row:** one pseudonymized content item for one client on one report date (`report_date × client_hash_id × content_hash_id`) in the daily performance fact table. The feature frame later aggregates the March daily rows to one row per client-content pair.
2. **Tables:** `fact_content_daily_performance` for daily Google Search Console (GSC) metrics; no join to client names, URLs, or raw queries.
3. **Time window:** March 1–31, 2026 for features. April 1–30, 2026 is used only as the future outcome window to define the proxy label. March is a mid-panel development month; June 2026 remains sealed and is not used here.
4. **Target/proxy:** `label_decline_20pct_april = 1` when April's available GSC impressions are at least 20% lower than March's impressions; otherwise 0. This is a directional proxy, not a measure of content quality or a causal effect. Items need at least 100 available March impressions and available GSC data in April to be labeled.
5. **Deliberately excluded:** `report_date`-level April metrics and any April-derived columns are excluded from features because they are only known after the March decision moment. Pseudonymous IDs are join/group context, never model features.

**Decision moment:** after March closes. A useful output is a ranked review queue of eligible pages, with the editor making the final decision.


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Reason |
|---|---|---|
| Features (March only) | `impressions_march`, `clicks_march`, `avg_position_march`, `days_with_impressions_march`, `ctr_pct_march` | Summarize observed GSC performance available by March 31. |
| Label / proxy | `label_decline_20pct_april` | Computed from April impressions compared with March; future outcome only. |
| Context | `client_hash_id`, `content_hash_id`, `report_date` | Identify, group, and aggregate rows; IDs must not be learned by the model. |
| Excluded | April impressions/clicks/position, availability flags as model inputs, raw URLs/names/queries, any label copy | Future information, measurement-status metadata, or identifying data; not valid predictive signals for this decision. |

**Five-feature availability notes:**
- `impressions_march`: knowable because the March GSC impression total is observed by March 31.
- `clicks_march`: knowable because the March GSC click total is observed by March 31.
- `avg_position_march`: knowable because it is averaged from March daily GSC positions; zero placeholders are treated as missing.
- `days_with_impressions_march`: knowable because it counts March dates with positive impressions.
- `ctr_pct_march`: knowable because it is calculated from March clicks divided by March impressions (scaled by 100); it uses no April data.


## 3. Connect to the gated warehouse

This setup uses the Colab Secret `HF_TOKEN`. If you see an access/401/403 error, confirm that you accepted the dataset terms in the browser and that the token is a Read token with access to this gated dataset. Do not put the token into the notebook source.


In [6]:
%pip -q install duckdb scikit-learn pandas


In [7]:
import os
import pandas as pd
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. In Colab, open the Secrets (key) panel, add HF_TOKEN, "
        "paste your Hugging Face READ token there, and enable notebook access. Do not paste it into a code cell."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
print("DuckDB connected. March and April 2026 partitions are configured.")


DuckDB connected. March and April 2026 partitions are configured.


### Verification query 1 of 3 — grain

Expected grain: one row per `report_date × client_hash_id × content_hash_id`. This query counts duplicate key groups; **0 means the declared grain is supported** for this month.


In [8]:
grain_check = con.sql(f"""
    SELECT COUNT(*) AS duplicate_grain_groups
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {MARCH}
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING COUNT(*) > 1
    ) duplicates
""").df()
display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_grain_groups
0,0


### Verification query 2 of 3 — row count and date span

This verifies the size and actual dates in the March slice rather than assuming the partition name proves its contents.


In [9]:
slice_check = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_report_date,
           MAX(report_date) AS max_report_date
    FROM {MARCH}
""").df()
display(slice_check)


,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


### Verification query 3 of 3 — availability

Availability flags are three-valued: `TRUE`, `FALSE`, or `NULL`. Use `IS TRUE` so only explicitly available GSC rows survive; do not assume NULL means FALSE or that zero-filled metrics mean measured zero.


In [10]:
availability_check = con.sql(f"""
    SELECT COUNT(*) AS all_march_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_is_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS rows_not_explicitly_available
    FROM {MARCH}
""").df()
display(availability_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_march_rows,rows_surviving_is_true,rows_not_explicitly_available
0,9841378,3611061,6230317


## 4. Build the five-feature frame and future proxy label

The features are aggregated from March rows with `gsc_data_available IS TRUE`. The label is built from April's available impressions, so it represents a future outcome relative to the March decision moment. The SQL brings only the page-level aggregate into pandas, not the raw daily warehouse.


In [11]:
feature_sql = f"""
WITH march_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_march,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_march,
        100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_pct_march
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
),
april_outcomes AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions_march,
    m.clicks_march,
    m.avg_position_march,
    m.days_with_impressions_march,
    m.ctr_pct_march,
    a.impressions_april,
    CASE WHEN a.impressions_april < 0.8 * m.impressions_march THEN 1 ELSE 0 END AS label_decline_20pct_april
FROM march_features m
INNER JOIN april_outcomes a USING (client_hash_id, content_hash_id)
"""

data = con.sql(feature_sql).df()
feature_cols = [
    "impressions_march",
    "clicks_march",
    "avg_position_march",
    "days_with_impressions_march",
    "ctr_pct_march",
]
label_col = "label_decline_20pct_april"

print(f"Feature frame: {len(data):,} eligible client-content items; {len(feature_cols)} features.")
print("Label prevalence:", round(data[label_col].mean(), 3) if len(data) else "no labeled rows")
display(data[["client_hash_id", "content_hash_id"] + feature_cols + [label_col]].head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 100,893 eligible client-content items; 5 features.
Label prevalence: 0.515


,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,days_with_impressions_march,ctr_pct_march,label_decline_20pct_april
0,client_62f4a7e64f5e0096,content_f1c085e5ea530266,207.0,0.0,5.979673,31,0.000000,1
1,client_62f4a7e64f5e0096,content_c4901b51a12b1c3b,218.0,1.0,8.071242,31,0.458716,1
2,client_62f4a7e64f5e0096,content_e68b30ce53db783f,150.0,1.0,4.478327,29,0.666667,1
3,client_62f4a7e64f5e0096,content_c43f069ed25dd56e,291.0,2.0,6.293391,31,0.687285,1
4,client_62f4a7e64f5e0096,content_a77ceb7afcf6ec78,938.0,3.0,2.913256,31,0.319829,1


### Why these five features are available at decision time

- **`impressions_march`** — the sum of observed March impressions; available after March closes.
- **`clicks_march`** — the sum of observed March clicks; available after March closes.
- **`avg_position_march`** — average of March daily average positions, excluding zero placeholders; available after March closes.
- **`days_with_impressions_march`** — number of distinct March days with positive impressions; available after March closes.
- **`ctr_pct_march`** — March clicks divided by March impressions × 100; calculated only from March values.

The label is not a feature. April outcomes are used only to evaluate whether the March signals help prioritize pages for later review.


## 5. Deliberate leakage experiment

First score a small decision tree using only the five honest March features. Then intentionally add a column that is an exact copy of the April-derived label. That second score should jump toward 1.00 because the answer has been smuggled into the inputs. **It is a demonstration of leakage, not a valid model.** Finally, remove the leak column and explicitly keep the five-feature list.


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

model_data = data.dropna(subset=feature_cols + [label_col]).copy()
if len(model_data) < 20 or model_data[label_col].nunique() < 2:
    raise ValueError(
        "Not enough labeled examples or only one target class in this slice. "
        "Check HF access, availability filtering, and the month window before interpreting scores."
    )

train_idx, test_idx = train_test_split(
    model_data.index, test_size=0.25, random_state=42, stratify=model_data[label_col]
)
y_train = model_data.loc[train_idx, label_col]
y_test = model_data.loc[test_idx, label_col]

honest_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42)
honest_model.fit(model_data.loc[train_idx, feature_cols], y_train)
honest_pred = honest_model.predict(model_data.loc[test_idx, feature_cols])
honest_accuracy = accuracy_score(y_test, honest_pred)
honest_balanced_accuracy = balanced_accuracy_score(y_test, honest_pred)

# INTENTIONAL LEAK: this input directly copies the target label. Never use it in a real model.
leaky_train = model_data.loc[train_idx, [label_col]].rename(columns={label_col: "leak_label_copy"})
leaky_test = model_data.loc[test_idx, [label_col]].rename(columns={label_col: "leak_label_copy"})
leaky_model = DecisionTreeClassifier(max_depth=1, random_state=42)
leaky_model.fit(leaky_train, y_train)
leaky_pred = leaky_model.predict(leaky_test)
leaky_accuracy = accuracy_score(y_test, leaky_pred)
leaky_balanced_accuracy = balanced_accuracy_score(y_test, leaky_pred)

score_comparison = pd.DataFrame([
    {"experiment": "Honest March features", "accuracy": honest_accuracy, "balanced_accuracy": honest_balanced_accuracy},
    {"experiment": "LEAK DEMO: label copied into input", "accuracy": leaky_accuracy, "balanced_accuracy": leaky_balanced_accuracy},
])
display(score_comparison.round(3))
print("The leaky score is invalid because the input contains the answer being predicted.")

# Remove the leak column: retain only the honest, pre-decision feature list.
X_honest = model_data[feature_cols].copy()
assert "leak_label_copy" not in X_honest.columns
assert X_honest.columns.tolist() == feature_cols
print("Leak removed. Final feature columns:", X_honest.columns.tolist())


,experiment,accuracy,balanced_accuracy
0,Honest March features,0.621,0.616
1,LEAK DEMO: label copied into input,1.000,1.000


The leaky score is invalid because the input contains the answer being predicted.
Leak removed. Final feature columns: ['impressions_march', 'clicks_march', 'avg_position_march', 'days_with_impressions_march', 'ctr_pct_march']


## 6. Data limitation

**Named limitation — unbalanced tracking history and three-valued availability.** Clients began GSC/GA4 tracking at different times, and availability can be `TRUE`, `FALSE`, or `NULL`. This notebook includes only rows explicitly marked `gsc_data_available IS TRUE`, and the April label requires available April data. Therefore, the resulting page set may not represent all content or all clients; results should be described as directional decision support for this eligible slice.

Additional caution: the 20% decline label is a hand-chosen proxy, not a ground-truth measure of content quality. Seasonality, algorithm updates, and other external changes may affect impressions. A future model should be compared with a simple baseline and validated across time and clients.


## Self-check

- [ ] The contract states the grain, tables, time windows, proxy label, and an explicit exclusion.
- [ ] Exactly three verification queries are present, with outputs shown after **Runtime → Run all**.
- [ ] Availability uses `gsc_data_available IS TRUE`.
- [ ] The feature frame has at most five pre-decision features and a one-line availability explanation for each.
- [ ] The deliberate leak experiment is shown, labeled invalid, and removed from the final feature set.
- [ ] No token, client name, URL, or private query is printed or committed.
- [ ] The executed notebook is committed as `work/notebooks/w03_data_contract.ipynb`.

